# Data Cleansing

## Data metadata

### [Natural Disaster Trends](https://zenodo.org/records/7930540)

**Disaster_Group:** EM-DAT stores different types of disasters: natural, technological and complex. The dataset has been filtered to contain only natural disasters so this column is "Natural" for all rows but kept for compatibility reasons.

**Disaster_Subgroup:** Every natural disaster is assigned to one of the following six subgroups: Biological, Geophysical, Climatological, Hydrological, Meteorological and Extra-terrestrial to describe the type of natural disaster. No missing values are present for this attribute.

**Disaster_type:** For every natural disaster event one main disaster type is identified. If two or more disasters are related because they are consequences of each other, then this information is encoded in the attributes Associated_Dis and Associated_Dis2. No missing values are present for this attribute.

**Disaster Sub-Type:** Subdivision related to the attribute Disaster_type so that a the disaster type Storm can be further classified as tropical, extra-tropical or convective storm.

**Disaster Sub-Sub Type:** Any appropriate sub-division of the disaster sub-type (not applicable for all disaster sub-types). Types of natural disasters could be further broken down using two more categories which would be available in the database. For example, the Disaster type Storm could be further subdivided into Tropical storm, Extra-tropical storm or Convective storm. Even a further subdivision of the category Convective storm would be possible. Since the analysis is aimed at detecting trends on a high level, the classification of each event based on the attributes Disaster_Subgroup and Disaster _Type was considered sufficient and the further subdivisions into Disaster sub-type and Disaster Subsubtype is only intended to be considered for detailed analysis. The full table is shown in the Appendix.

**Associated_Dis:** Secondary event triggered by a natural disaster (i.e. Landslide for a flood, explosion after an earthquake, ...)

**Associated_Dis2:** Another secondary event triggered by a natural disaster. (i.e. Landslide for a flood, explosion after an earthquake, ...)

Example: If a tsunami is triggered by an earthquake, then the attribute Disaster_Type would be Earthquake, the attribute Disaster_Subtype would be Ground movement and the attribute Associated_Dis would be Tsunami/Tidal wave.

**Country:** The country in which the disaster has occurred or had an impact. If a disaster has affected more than one country, a seperate entry is created in the database for each country affected. No missing values are present for this attribute.

**ISO:** Unique 3-letter code for each country defined by ISO 3166. No missing values are present for this attribute.

**Region:** The region to which the country belongs, based on the UN regional division. No missing values are present for this attribute.

**Continent:** The continent to which the country belongs. No missing values are present for this attribute.

**Start_Year:** The year when the disaster occurred. No missing values are present for this attribute.

**End Year:** The year when the disaster ended. No missing values are present for this attribute. For sudden-impact disasters also the month and the day are well defined and available. For disaster situations developing gradually over a longer time period (i.e. drought) with no specific start date the day attribute is empty. For our questions the exact date plays a subordinate role and therefore the year of the beginning of the disaster is completely sufficient for our analysis.

**Total_Deaths:** Number of people who lost their life because the event happened plus the number of people whose whereabouts since the disaster are unknown, and presumed dead based on official figures. Missing values present for approx. 25% of all events.

**No_Affected:** Number of people which requiring immediate assistance during an emergency situation. The indicator affected is often reported and is widely used by different actors to convey the extent, impact, or severity of a disaster in non-spatial terms. In case that no values for the attribute Total_Deaths are available this attribute could be used as a proxy.

### [Global Spread of Conflict by Country and Population](https://datacatalog.worldbank.org/search/dataset/0041070/Global-Spread-of-Conflict-by-Country-and-Population)

This dataset provides the spread of the conflict globally in terms of population and country for the years 2000-2016.

```Fields: TODO```

### [World Happiness Report](https://worldhappiness.report/data/)

```Fields: TODO```

### [Climate Change: Earth Surface Temperature Data](https://www.kaggle.com/datasets/berkeleyearth/climate-change-earth-surface-temperature-data)

**Date:** starts in 1750 for average land temperature and 1850 for max and min land temperatures and global ocean and land temperatures


**LandAverageTemperature:** global average land temperature in celsius
LandAverageTemperatureUncertainty: the 95% confidence interval around the average

**LandMaxTemperature:** global average maximum land temperature in celsius
LandMaxTemperatureUncertainty: the 95% confidence interval around the maximum land temperature

**LandMinTemperature:** global average minimum land temperature in celsius
LandMinTemperatureUncertainty: the 95% confidence interval around the minimum land temperature

**LandAndOceanAverageTemperature:** global average land and ocean temperature in celsius
LandAndOceanAverageTemperatureUncertainty: the 95% confidence interval around the global average land and ocean temperature

In [ ]:
# Imports

import pandas as pd

### disaster.csv

In [ ]:
# load disaster.csv

disaster_df = pd.read_csv('../data/disaster.csv')

In [ ]:
# get insights on data

display(disaster_df.shape)

display(disaster_df.dtypes)

display(disaster_df.count())

In [ ]:
# add columname to the first column (Unnamed) and make it the index
disaster_df.rename(columns={"Unnamed: 0":"id"}, inplace=True)
disaster_df.set_index("id",inplace=True)

# remove columns with unnnecessary or highly missing data
disaster_df.drop(columns=["Disaster_Subtype","Disaster_Subsubtype","Associated_Dis","Associated_Dis2","Disaster_Decade"],inplace=True)

In [ ]:
# replace NaNs

# replace NaNs in Total_Deaths col with 0s, because the dataset will show how many deaths are we aware of
disaster_df["Total_Deaths"] = disaster_df["Total_Deaths"].fillna(0)

# replace NaNs in Total_Affected col with the death number, because at least that amount of people were affected
disaster_df["Total_Affected"] = disaster_df["Total_Affected"].fillna(disaster_df["Total_Deaths"])

# fix the types of the columns
disaster_df = disaster_df.convert_dtypes()

In [ ]:
# observe the data again

display(disaster_df.shape)

display(disaster_df.dtypes)

display(disaster_df.count())

### create the countries dataframe

In [ ]:
# create a new dataframe to store the country names, continents and ISO codes there
countries_df = disaster_df[["ISO","Country","Continent"]].set_index("ISO")
countries_df.drop_duplicates(inplace=True)
countries_df.rename(columns={"Country":"Name"}, inplace=True)

# remove the redundant country name and continent columns from disaster dataframe
disaster_df.drop(columns=["Country","Continent"], inplace=True)

In [ ]:
custom_country_map = {
    "Congo (Brazzaville)": "Congo",  # Republic of the Congo
    "Congo (Kinshasa)": "Democratic Republic of the Congo",
    "Hong Kong S.A.R. of China": "Hong Kong",
    "Ivory Coast": "C?te d'Ivoire",
    "Russia": "Russian Federation (the)",
    "State of Palestine": "Palestine, State of",
    "Taiwan Province of China": "Taiwan",
    "Macedonia": "North Macedonia",  # Now officially North Macedonia
    "Turks And Caicas Islands": "Turks and Caicos Islands",
    "Virgin Islands": "Virgin Islands, U.S.",  # Could also be Virgin Islands, British
    "Europe": None,  # Europe is not a country
    "North America": None,  # Continent, not a country
    "South America": None,  # Continent, not a country
    "Reunion": "R?union",  # Official name with accent
    "South Georgia And The South Sandwich Isla": "South Georgia and the South Sandwich Islands",
    "Swaziland": "Eswatini",  # Renamed to Eswatini
    "Timor Leste": "Timor-Leste",  # Official name
    "Gaza Strip": "Palestine",
    "Asia": None,  # Continent, not a country
    "Burma": "Myanmar",  # Official name is Myanmar
    "Palestina": "Palestine",
    "Bonaire, Saint Eustatius And Saba": "Bonaire, Sint Eustatius and Saba",  # Special municipality
    "Oceania": None,  # Continent, not a country
    "Cape Verde": "Cabo Verde",  # Official name
    "Guinea Bissau": "Guinea-Bissau",  # Official name with hyphen
    "Africa": None,  # Continent, not a country
    "Congo (Democratic Republic Of The)": "Democratic Republic of the Congo",  # Official name
    "Falkland Islands (Islas Malvinas)": "Falkland Islands",  # Recognized name
    "French Southern And Antarctic Lands": "French Southern Territories",  # Recognized name
}

custom_iso_codes = {
    "Kosovo": "XKX",  # Kosovo has the ISO code XKX, often used in informal settings
    "Somaliland region": "SOM",  # Associated with Somalia (SOM), despite self-declared independence
    "Russia": "RUS",  # Russia is a country
    "?land": "ALA",  # ISO code for ?land Islands
    "Baker Island": "BK",
    "Saint Martin": "MF",
    "Sint Maarten": "SX",
    "Palmyra Atoll": "PL",
    "Macau": "MO",
    "Kingman Reef": "KR",
}

# add the custom country names to the countries dataframe
for custom_name, official_name in custom_iso_codes.items():
    if official_name is not None:
        countries_df.loc[official_name] = custom_name

In [ ]:
# export the dataframes to csv files

disaster_df.to_csv("../cleaned_data/disaster_cleaned.csv")
countries_df.to_csv("../cleaned_data/countries.csv")

In [ ]:
def get_iso(name):
    res = countries_df[countries_df["Name"]==name].index 
    if len(res) == 0:
        return name
    else:
        return res[0]

In [ ]:
import pycountry

def get_iso_code(country_name):
    '''get the ISO code for each country'''

    try:
        return pycountry.countries.lookup(country_name).alpha_3
    except LookupError:
        if country_name in custom_country_map.keys():
            try:
                if custom_country_map[country_name] is None:
                    return None
                return pycountry.countries.lookup(custom_country_map[country_name]).alpha_3
            except LookupError:
                res = get_iso(custom_country_map[country_name])
        else:
            res = get_iso(country_name)
        
        if len(res) != 3:
            res = custom_iso_codes[country_name] if country_name in custom_iso_codes.keys() else None
            if res is None:
                print(f"Country {country_name} not found")
        
        return res

### happiness.xls

In [ ]:
# import file
df_happiness = pd.read_excel('../data/happiness.xls', parse_dates=True)

display(df_happiness.head())

display(df_happiness.dtypes)

display(df_happiness.count())


In [ ]:
# fill in the empty cells with 0
df_happiness.fillna(0, inplace=True)

display(df_happiness["Country name"].count())

df_happiness["ISO"] = df_happiness["Country name"].apply(get_iso_code)

df_happiness.drop(columns=["Country name"], inplace=True)

display(df_happiness.head())

display(df_happiness.dtypes)

display(df_happiness.count())

In [ ]:
#export as .csv
df_happiness.to_csv("../cleaned_data/happiness_cleaned.csv")

### temperature.csv

In [ ]:
#used the same function as in "happines" file
df_glbl = pd.read_csv('../data/temperature.csv', parse_dates=True)

display(df_glbl.head())

display(df_glbl.dtypes)

display(df_glbl.count())

In [ ]:
df_glbl.ffill(inplace=True)

# aggregate dt column to year
df_glbl['dt'] = pd.to_datetime(df_glbl['dt'])
df_glbl['Year'] = df_glbl['dt'].dt.year
df_glbl.drop(columns=["dt"], inplace=True)

df_glbl["Country"] = df_glbl["Country"].str.replace(" (Europe)", "")

df_glbl = df_glbl.groupby(['Year', 'Country'], as_index=False).agg({'AverageTemperature':'mean', 'AverageTemperatureUncertainty':'mean'})

df_glbl.drop_duplicates(inplace=True)

df_glbl.convert_dtypes()

In [ ]:
display(df_glbl["Country"].nunique())

df_glbl["ISO"]= df_glbl["Country"].apply(get_iso_code)

display(df_glbl["ISO"].nunique())

display(df_glbl.count())



In [ ]:
df_glbl.drop(columns=["Country"], inplace=True)
df_glbl.dropna(inplace=True)

In [ ]:
display(df_glbl.head())

display(df_glbl.dtypes)

display(df_glbl.count())

In [ ]:
#export as .csv file
df_glbl.to_csv("../cleaned_data/glbl_cleaned.csv")